<a href="https://colab.research.google.com/github/PARIJAAT-13/Flyrank-A.I/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PARIJAAT-13/Flyrank-A.I/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Ranked actions

1. **Refresh stale pages with strong demand signals**
   - Reason code: `STALE_HIGH_DEMAND`
   - Trigger: high `days_since_last_update` combined with strong search volume or impressions.
   - Action: review the page and prioritize it for a content refresh.

2. **Review pages with high impressions but weaker position**
   - Reason code: `HIGH_IMPRESSIONS_WEAK_POSITION`
   - Trigger: meaningful impressions with an average position that suggests room for improvement.
   - Action: human review for content relevance, structure, and search intent.

3. **Review pages with strong search demand**
   - Reason code: `HIGH_SEARCH_VOLUME`
   - Trigger: high search volume.
   - Action: use search volume as a prioritization signal, not as proof of expected traffic.

4. **Deprioritize pages with weak evidence**
   - Reason code: `LOW_SIGNAL`
   - Trigger: low demand and weak performance signals.
   - Action: do not prioritize until stronger evidence appears.

### Principle

The queue is a **decision-support ranking**, not an automatic instruction to change content. Every recommended action requires human review.

In [ ]:
import pandas as pd
from pathlib import Path

# Clone repository if needed
repo_root = Path("/content/Flyrank-A.I")

if not repo_root.exists():
    !git clone https://github.com/PARIJAAT-13/Flyrank-A.I.git /content/Flyrank-A.I

data_path = repo_root / "data/raw/content_refresh_anonymized.csv"

print("Repository:", repo_root.exists())
print("Dataset:", data_path.exists())

df = pd.read_csv(data_path)

for col in [
    "days_since_last_update",
    "search_volume",
    "impressions_last_30d",
    "avg_position"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df["staleness_score"] = df["days_since_last_update"].rank(pct=True)
df["volume_score"] = df["search_volume"].rank(pct=True)
df["impression_score"] = df["impressions_last_30d"].rank(pct=True)

df["priority_score"] = (
    0.40 * df["staleness_score"]
    + 0.30 * df["volume_score"]
    + 0.20 * df["impression_score"]
)

def reason_code(row):
    if row["staleness_score"] >= 0.75 and (
        row["volume_score"] >= 0.75 or row["impression_score"] >= 0.75
    ):
        return "STALE_HIGH_DEMAND"
    elif row["impression_score"] >= 0.75 and row["avg_position"] > 10:
        return "HIGH_IMPRESSIONS_WEAK_POSITION"
    elif row["volume_score"] >= 0.75:
        return "HIGH_SEARCH_VOLUME"
    else:
        return "LOW_SIGNAL"

df["reason_code"] = df.apply(reason_code, axis=1)

action_map = {
    "STALE_HIGH_DEMAND": "Review for content refresh",
    "HIGH_IMPRESSIONS_WEAK_POSITION": "Review content and search intent",
    "HIGH_SEARCH_VOLUME": "Review search-demand opportunity",
    "LOW_SIGNAL": "Deprioritize pending stronger evidence"
}

df["recommended_action"] = df["reason_code"].map(action_map)

action_queue = df.sort_values(
    "priority_score",
    ascending=False
).reset_index(drop=True)

print("Rows:", len(action_queue))
print("\nReason-code counts:")
print(action_queue["reason_code"].value_counts())

display(
    action_queue[
        ["priority_score", "reason_code", "recommended_action"]
    ].head(10)
)

Repository: True
Dataset: True
Rows: 30000

Reason-code counts:
reason_code
LOW_SIGNAL                        19304
HIGH_SEARCH_VOLUME                 4540
STALE_HIGH_DEMAND                  4447
HIGH_IMPRESSIONS_WEAK_POSITION     1709
Name: count, dtype: int64


,priority_score,reason_code,recommended_action
0,0.833433,STALE_HIGH_DEMAND,Review for content refresh
1,0.831155,STALE_HIGH_DEMAND,Review for content refresh
2,0.830112,STALE_HIGH_DEMAND,Review for content refresh
3,0.829402,STALE_HIGH_DEMAND,Review for content refresh
4,0.828272,STALE_HIGH_DEMAND,Review for content refresh
5,0.827673,STALE_HIGH_DEMAND,Review for content refresh
6,0.827290,STALE_HIGH_DEMAND,Review for content refresh
7,0.827158,STALE_HIGH_DEMAND,Review for content refresh
8,0.825515,STALE_HIGH_DEMAND,Review for content refresh
9,0.825425,STALE_HIGH_DEMAND,Review for content refresh


### Intended use

This playbook is intended to help a human content/SEO reviewer prioritize pages for review and possible refresh.

The recommendations are based on observed dataset signals such as freshness, search volume, impressions, and average position. They are decision-support signals, not guaranteed predictions of future traffic or rankings.

### Limits

- The model was evaluated using a grouped client split.
- Validation showed that the Random Forest did not outperform the Week-4 baseline.
- Search volume should not be treated as a guaranteed page-level traffic forecast.
- The recommendations do not establish that refreshing a page will cause better performance.
- The playbook is not intended for fully automated content changes or production decisions.

A human should review the page, search intent, business context, and content quality before taking action.

In [ ]:
# Section 2 — Intended use and limits

print("=== INTENDED USE ===")
print("Human-reviewed prioritization of pages for possible content actions.")

print("\n=== LIMITS ===")
limits = [
    "Decision-support only; not an automatic content-change system.",
    "Grouped validation showed the Random Forest did not outperform the baseline.",
    "Search volume is a prioritization signal, not a guaranteed traffic forecast.",
    "The results do not prove that refreshing content causes better performance.",
    "Human review is required before taking action."
]

for i, item in enumerate(limits, 1):
    print(f"{i}. {item}")

print("\n=== QUEUE SUMMARY ===")
print("Total pages:", len(action_queue))
print("Recommended actions are ranked by observed feature signals.")

=== INTENDED USE ===
Human-reviewed prioritization of pages for possible content actions.

=== LIMITS ===
1. Decision-support only; not an automatic content-change system.
2. Grouped validation showed the Random Forest did not outperform the baseline.
3. Search volume is a prioritization signal, not a guaranteed traffic forecast.
4. The results do not prove that refreshing content causes better performance.
5. Human review is required before taking action.

=== QUEUE SUMMARY ===
Total pages: 30000
Recommended actions are ranked by observed feature signals.


### Human review rules

Before taking any recommended action, a human reviewer should check:

- Search intent and whether the page actually matches it.
- Current content quality and relevance.
- Whether the freshness signal is meaningful for that topic.
- Search volume and impressions as supporting signals, not guarantees.
- Important business, editorial, or brand constraints.
- Whether the recommendation makes sense for the specific page.

### No-go list

The system should **not** automatically:

- Rewrite or publish content.
- Delete or redirect pages.
- Make claims about guaranteed ranking or traffic gains.
- Treat search volume as guaranteed future traffic.
- Treat correlation as proof that refreshing caused improvement.
- Make high-impact SEO or business decisions without human review.

The playbook is a prioritization tool, not an autonomous content-management system.

In [ ]:
# Section 3 — Human review + no-go checks

print("=== HUMAN REVIEW CHECK ===")

review_rules = [
    "Check search intent and content relevance.",
    "Check freshness and performance signals.",
    "Check search volume and impressions as supporting signals.",
    "Check business/editorial constraints.",
    "Human approval is required before action."
]

for i, rule in enumerate(review_rules, 1):
    print(f"{i}. {rule}")

print("\n=== NO-GO LIST ===")

no_go = [
    "Do not automatically rewrite or publish content.",
    "Do not automatically delete or redirect pages.",
    "Do not claim guaranteed traffic or ranking gains.",
    "Do not treat correlation as causation.",
    "Do not make high-impact decisions without human review."
]

for i, rule in enumerate(no_go, 1):
    print(f"{i}. {rule}")

print("\n✅ Human review and no-go rules defined.")

=== HUMAN REVIEW CHECK ===
1. Check search intent and content relevance.
2. Check freshness and performance signals.
3. Check search volume and impressions as supporting signals.
4. Check business/editorial constraints.
5. Human approval is required before action.

=== NO-GO LIST ===
1. Do not automatically rewrite or publish content.
2. Do not automatically delete or redirect pages.
3. Do not claim guaranteed traffic or ranking gains.
4. Do not treat correlation as causation.
5. Do not make high-impact decisions without human review.

✅ Human review and no-go rules defined.


### Intended use

This playbook is a decision-support tool for prioritizing pages for human review.

It uses observed signals such as freshness, search volume, impressions, and average position to rank pages and provide a reason code.

The queue helps a content team decide which pages may deserve attention first. It does not predict guaranteed traffic gains or prove that a recommended action will improve performance.

### Limits

- Results are directional, not causal.
- The model was evaluated on a grouped client split, so performance may differ on new data.
- Recommendations depend on the available dataset and feature definitions.
- Human review is required before any content change.
- The system should not automatically publish, delete, rewrite, or make strategic SEO decisions.

In [ ]:
# Section 2 — Intended use and limits

print("=== INTENDED USE ===")
print("Purpose: prioritize pages for human review.")
print("Output: ranked actions + reason codes.")
print("Use: decision support, not automatic execution.")

print("\n=== LIMITS ===")
limits = [
    "Directional, not causal",
    "Performance may differ on new data",
    "Recommendations depend on available features",
    "Human review required",
    "No automatic publishing or content changes"
]

for limit in limits:
    print("-", limit)

print("\n✅ Intended-use and limits check complete.")

=== INTENDED USE ===
Purpose: prioritize pages for human review.
Output: ranked actions + reason codes.
Use: decision support, not automatic execution.

=== LIMITS ===
- Directional, not causal
- Performance may differ on new data
- Recommendations depend on available features
- Human review required
- No automatic publishing or content changes

✅ Intended-use and limits check complete.


### Monitoring / retrain triggers

The recommendation queue should be reviewed periodically rather than treated as permanent.

Monitor:
- Feature distributions for major changes in freshness, search volume, impressions, and position.
- The distribution of reason codes and priority scores.
- Validation MAE/RMSE when new labeled data becomes available.
- Whether the highest-priority recommendations remain useful after human review.

Retraining should be considered if the feature distribution changes materially, validation error worsens consistently, or the recommendation patterns become stale.

These are monitoring and decision-support triggers, not automatic production retraining rules.

In [ ]:
# Section 4 — Monitoring / retrain checks

print("=== MONITORING / RETRAIN TRIGGERS ===")

print("1. Monitor feature distributions for drift.")
print("2. Monitor reason-code and priority-score distributions.")
print("3. Recheck validation MAE/RMSE when new labels are available.")
print("4. Review whether high-priority recommendations remain useful.")
print("5. Consider retraining when drift or validation degradation is persistent.")

print("\nStatus: Monitoring framework defined.")
print("No automatic production retraining is recommended.")

=== MONITORING / RETRAIN TRIGGERS ===
1. Monitor feature distributions for drift.
2. Monitor reason-code and priority-score distributions.
3. Recheck validation MAE/RMSE when new labels are available.
4. Review whether high-priority recommendations remain useful.
5. Consider retraining when drift or validation degradation is persistent.

Status: Monitoring framework defined.
No automatic production retraining is recommended.


### Exports for the paper

The ranked action queue is exported to `work/outputs/` so it can be regenerated from the notebook and reused in the research paper.

In [ ]:
# Section 5 — Export ranked queue

from pathlib import Path

output_dir = repo_root / "work/outputs"
output_dir.mkdir(parents=True, exist_ok=True)

queue_path = output_dir / "w07_ranked_action_queue.csv"

export_columns = [
    "priority_score",
    "reason_code",
    "recommended_action"
]

action_queue[export_columns].to_csv(
    queue_path,
    index=False
)

print("=== EXPORT ===")
print("Queue rows:", len(action_queue))
print("Export path:", queue_path)
print("File exists:", queue_path.exists())

print("\n✅ Ranked action queue exported successfully.")

=== EXPORT ===
Queue rows: 30000
Export path: /content/Flyrank-A.I/work/outputs/w07_ranked_action_queue.csv
File exists: True

✅ Ranked action queue exported successfully.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.